In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAI
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.prompts import PromptTemplate
from langchain.schema import HumanMessage
# # Specify the path to the .env file in the other directory
dotenv_path = '../.env'

# Load the environment variables from the specified .env file
load_dotenv(dotenv_path=dotenv_path)
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
os.getenv("GEMINI_API_KEY")
print(gemini_api_key)
# model_id = "gemini-2.5-flash-lite"
model_id = "gemini-2.5-pro"
llm = GoogleGenerativeAI(model=model_id, google_api_key=gemini_api_key,temperature=0.0) 

In [12]:
response = llm.invoke("Hello there")
print(response)

Hello there! How can I help you today?


## Designing our Agent's Memory

In [3]:
class State(TypedDict):
    text: str
    classification : str
    entities : list[str]
    summary: str


## Input Classification

In [5]:
def classification_node(state: State):
    ''' Classify the text into one of the categories: News, Blog, Research, or Other '''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Classify the following text into one of the categories: News, Blog, Research, or Other.\n\nText:{text}\n\nCategory:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    classification = llm.invoke([message]).content.strip()
    return {"classification": classification}

In [17]:
# ''' Classify the text into one of the categories: News, Blog, Research, or Other '''
# prompt = PromptTemplate(
#     input_variables=["text"],
#     template="Classify the following text into one of the categories: News, Blog, Research, or Other.\n\nText:{text}\n\nCategory:"
# )
# message = HumanMessage(content=prompt.format(text="Moneky"))
# classification = llm.invoke([message]).content()
print(type(classification))

<class 'str'>


## Entity Extraction

In [7]:
def entity_extraction_node(state: State):
    ''' Extract all the entities (Person, Organization, Location) from the text      '''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Extract all the entities (Person, Organization, Location) from the following text. Provide the result as a comma-separated list.\n\nText:{text}\n\nEntities:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    entities = llm.invoke([message]).content.strip().split(", ")
    return {"entities": entities}

## Summarization

In [8]:
def summarization_node(state: State):
    ''' Summarize the text in one short sentence '''
    prompt = PromptTemplate(
        input_variables=["text"],
        template="Summarize the following text in one short sentence.\n\nText:{text}\n\nSummary:"
    )
    message = HumanMessage(content=prompt.format(text=state["text"]))
    summary = llm.invoke([message]).content.strip()
    return {"summary": summary}

## Creating the Workflow

#### Start -> Classification Node -> Extraction Node -> Summarization Node -> END

In [9]:
workflow = StateGraph(State)

# Adding Nodes
workflow.add_node("classification_node", classification_node)
workflow.add_node("entity_extraction", entity_extraction_node)
workflow.add_node("summarization", summarization_node)

# Add edges to the graph
# Set entrypoint to the Graph
workflow.set_entry_point("classification_node")
workflow.add_edge("classification_node", "entity_extraction")
workflow.add_edge("entity_extraction", "summarization")
workflow.add_edge("summarization", END)

# Compile the Graph
app = workflow.compile()

## Run that shit

In [10]:
sample_text = """
OpenAI has announced the GPT-4 model, which is a large multimodal model that exhibits human-level performance on various professional benchmarks. It is developed to improve the alignment and safety of AI systems.
additionally, the model is designed to be more efficient and scalable than its predecessor, GPT-3. The GPT-4 model is expected to be released in the coming months and will be available to the public for research and development purposes.
"""

In [ ]:
state_input = {"text": sample_text}
result = app.invoke(state_input)
print("Classification:", result["classification"])
print("\nEntities:", result["entities"])
print("\nSummary:", result["summary"])